<a href="https://colab.research.google.com/github/belinatom/NALAPROJECT/blob/main/nlp_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 — MLM Domain Adaptation

**Author**: Belinda Mziray | HSLU MSc IT, Digitalization & Sustainability | 2026

## Approach
1. **Stage 1**: Pre-train AfriBERTa (and SwahBERT) on Swahili proverbs using Masked Language Modeling (15% mask probability)
2. **Stage 2**: Fine-tune the adapted model for 56-class classification
3. **Compare**: Baseline (no MLM) vs MLM-adapted




In [ ]:
# Install & imports
!pip install -q gdown wandb aim mlflow

import os, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW

from transformers import (
    AutoTokenizer, AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import gdown

seed = 42
np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_style('whitegrid')
print(f'✓ Imports loaded | Device: {device}')


In [ ]:
# Download dataset from Google Drive
FILE_ID = '1Rp9Fg1BitlAR3d1JyH4DmTp3aSKr8H1t'
gdown.download(
    f'https://drive.google.com/uc?id={FILE_ID}',
    '/content/swahiliproverbs.csv',
    quiet=False, fuzzy=True
)

df = pd.read_csv('/content/swahiliproverbs.csv')
df = df[['swahili_proverb', 'label']].dropna().copy()
df['swahili_proverb'] = df['swahili_proverb'].astype(str).str.strip()
df['label']           = df['label'].astype(str).str.strip()
df = df[df['swahili_proverb'] != ''].reset_index(drop=True)

text_col, label_col = 'swahili_proverb', 'label'
print(f'✓ Loaded: {len(df)} proverbs, {df[label_col].nunique()} categories')
df.head(3)


In [ ]:
# Encode labels and split data
le = LabelEncoder()
y = le.fit_transform(df[label_col])
x = df[text_col].values
num_classes = len(le.classes_)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=seed, stratify=y
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.1, random_state=seed, stratify=y_train
)
print(f'✓ Train: {len(x_train)} | Val: {len(x_val)} | Test: {len(x_test)} | Classes: {num_classes}')


In [ ]:
# Config, MLM dataset, classification DataLoader
MAX_LENGTH = 64
BATCH_SIZE = 32
MLM_EPOCHS = 1
MLM_LR     = 5e-5

class MLMDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=MAX_LENGTH):
        self.encodings = tokenizer(
            list(texts), max_length=max_length,
            truncation=True, padding='max_length', return_tensors='pt'
        )
    def __len__(self):
        return self.encodings['input_ids'].shape[0]
    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx]
        }

def make_cls_loader(texts, labels, tokenizer, shuffle=False):
    enc = tokenizer(list(texts), max_length=MAX_LENGTH,
                    truncation=True, padding=True, return_tensors='pt')
    ds = torch.utils.data.TensorDataset(
        enc['input_ids'], enc['attention_mask'],
        torch.tensor(labels, dtype=torch.long)
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

print('✓ Dataset/loader helpers defined')


In [ ]:
# MLM pre-training function
def mlm_pretrain(model_name, output_dir):
    print(f'\n{"="*60}\nMLM PRE-TRAINING: {model_name}\n{"="*60}')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    mlm_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)

    # Use ALL proverbs (unsupervised — labels not needed)
    full_corpus = np.concatenate([x_train, x_val, x_test])

    mlm_dataset   = MLMDataset(full_corpus, tokenizer)
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=True, mlm_probability=0.15
    )
    mlm_loader = DataLoader(mlm_dataset, batch_size=BATCH_SIZE,
                            shuffle=True, collate_fn=data_collator)

    optimizer = AdamW(mlm_model.parameters(), lr=MLM_LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(mlm_loader) * MLM_EPOCHS)

    mlm_history = []
    for epoch in range(MLM_EPOCHS):
        mlm_model.train()
        total_loss = 0
        for batch in mlm_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            loss = mlm_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(mlm_loader)
        mlm_history.append(avg_loss)
        print(f'  Epoch {epoch+1}/{MLM_EPOCHS} | MLM Loss: {avg_loss:.4f}')

    mlm_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f'✓ MLM model saved to {output_dir}')
    return mlm_history

print('✓ MLM pre-training function defined')


In [ ]:
# Classification fine-tuning function
def train_classifier(model_path, label, num_epochs=5, lr=2e-5):
    print(f'\n{"="*60}\nCLASSIFICATION: {label}\n{"="*60}')
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    tr_loader = make_cls_loader(x_train, y_train, tokenizer, shuffle=True)
    vl_loader = make_cls_loader(x_val,   y_val,   tokenizer)
    te_loader = make_cls_loader(x_test,  y_test,  tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, num_labels=num_classes, ignore_mismatched_sizes=True
    ).to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(tr_loader) * num_epochs)

    best_val_acc, best_val_f1 = 0, 0
    history = {'epoch': [], 'train_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for input_ids, attention_mask, labels in tr_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
        train_loss = total_loss / len(tr_loader)

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for input_ids, attention_mask, labels in vl_loader:
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                targets.extend(labels.numpy())

        val_acc = accuracy_score(targets, preds)
        val_f1  = f1_score(targets, preds, average='macro', zero_division=0)
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['val_acc'].append(val_acc); history['val_f1'].append(val_f1)

        if val_acc > best_val_acc:
            best_val_acc, best_val_f1 = val_acc, val_f1
            torch.save(model.state_dict(), f'/tmp/best_cls_{label}.pth')

        print(f'  Epoch {epoch+1}/{num_epochs} | Loss={train_loss:.4f} | Val Acc={val_acc:.4f} F1={val_f1:.4f}')

    model.load_state_dict(torch.load(f'/tmp/best_cls_{label}.pth'))
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in te_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            targets.extend(labels.numpy())

    test_acc = accuracy_score(targets, preds)
    test_f1  = f1_score(targets, preds, average='macro', zero_division=0)
    print(f'\n✓ {label}: Val Acc={best_val_acc:.4f} F1={best_val_f1:.4f}')
    print(f'✓ {label}: Test Acc={test_acc:.4f} F1={test_f1:.4f}')

    return {'tag': label, 'val_acc': best_val_acc, 'val_f1': best_val_f1,
            'test_acc': test_acc, 'test_f1': test_f1,
            'history': history, 'model_path': f'/tmp/best_cls_{label}.pth'}

print('✓ Classification training function defined')


In [ ]:
# AfriBERTa: MLM pre-training + classification
AFRIBERTA_MODEL = 'castorini/afriberta_large'
AFRIBERTA_MLM_DIR = '/tmp/afriberta_mlm_adapted'

# Stage 1: MLM adaptation
mlm_history_afri = mlm_pretrain(AFRIBERTA_MODEL, AFRIBERTA_MLM_DIR)

# Stage 2a: Baseline (no MLM)
result_afri_base = train_classifier(AFRIBERTA_MODEL, 'AfriBERTa-Baseline', num_epochs=5, lr=2e-5)

# Stage 2b: With MLM adaptation
result_afri_mlm  = train_classifier(AFRIBERTA_MLM_DIR, 'AfriBERTa-MLM', num_epochs=5, lr=2e-5)


In [ ]:
# SwahBERT: download, MLM pre-training, classification
SWAHBERT_PATH    = '/content/swahbert'
SWAHBERT_MLM_DIR = '/tmp/swahbert_mlm_adapted'

if not os.path.exists(f'{SWAHBERT_PATH}/config.json'):
    os.makedirs(SWAHBERT_PATH, exist_ok=True)
    gdown.download_folder(
        'https://drive.google.com/drive/folders/1cCcPopqTyzY6AnH9quKcT9kG5zH7tgEZ',
        output=SWAHBERT_PATH, quiet=False
    )
    if os.path.exists(f'{SWAHBERT_PATH}/swahbert_config.json'):
        shutil.copy(f'{SWAHBERT_PATH}/swahbert_config.json', f'{SWAHBERT_PATH}/config.json')

# Stage 1: MLM adaptation
mlm_history_swah = mlm_pretrain(SWAHBERT_PATH, SWAHBERT_MLM_DIR)

# Stage 2a: Baseline
result_swah_base = train_classifier(SWAHBERT_PATH,    'SwahBERT-Baseline', num_epochs=5, lr=2e-5)

# Stage 2b: With MLM
result_swah_mlm  = train_classifier(SWAHBERT_MLM_DIR, 'SwahBERT-MLM',      num_epochs=5, lr=2e-5)


In [ ]:
# Results summary and visualization
all_results = [result_afri_base, result_afri_mlm, result_swah_base, result_swah_mlm]

results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('history', 'model_path')}
    for r in all_results
])[['tag', 'val_acc', 'val_f1', 'test_acc', 'test_f1']]

print('\n' + '='*60)
print('TASK 3 — FINAL RESULTS')
print('='*60)
print(results_df.to_string(index=False))

afri_acc_gain = (result_afri_mlm['test_acc'] - result_afri_base['test_acc']) * 100
afri_f1_gain  = (result_afri_mlm['test_f1']  - result_afri_base['test_f1'])  * 100
swah_acc_gain = (result_swah_mlm['test_acc'] - result_swah_base['test_acc']) * 100
swah_f1_gain  = (result_swah_mlm['test_f1']  - result_swah_base['test_f1'])  * 100

print(f'\nMLM effect on AfriBERTa: Accuracy {afri_acc_gain:+.2f}% | F1 {afri_f1_gain:+.2f}%')
print(f'MLM effect on SwahBERT : Accuracy {swah_acc_gain:+.2f}% | F1 {swah_f1_gain:+.2f}%')

results_df.to_csv('/content/task3_results.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(results_df['tag'], results_df['test_acc'],
            color=['#3498db', '#2ecc71', '#e67e22', '#f39c12'])
axes[0].set_title('Test Accuracy: Baseline vs MLM', fontweight='bold')
axes[0].set_ylabel('Accuracy'); axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(results_df['tag'], results_df['test_f1'],
            color=['#3498db', '#2ecc71', '#e67e22', '#f39c12'])
axes[1].set_title('Test F1: Baseline vs MLM', fontweight='bold')
axes[1].set_ylabel('F1'); axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('/content/task3_results.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Experiment Tracking — Logged at the End

The next cell logs all metrics, models, and artifacts to W&B, Aim, and MLflow. The main task code is uncontaminated.


In [ ]:
# Cell 10 — Log everything to W&B, Aim, and MLflow
from getpass import getpass
import wandb, aim, mlflow

CONFIG = {
    'task': 'Task 3',
    'mlm_epochs': MLM_EPOCHS, 'mlm_lr': MLM_LR, 'mlm_probability': 0.15,
    'cls_epochs': 5, 'cls_lr': 2e-5,
    'max_length': MAX_LENGTH, 'batch_size': BATCH_SIZE,
    'num_classes': num_classes,
    'train_size': len(x_train), 'val_size': len(x_val), 'test_size': len(x_test)
}

os.makedirs('/content/.aim', exist_ok=True)
os.makedirs('/content/mlruns', exist_ok=True)
mlflow.set_tracking_uri('file:///content/mlruns')

wandb_key = getpass('Paste W&B API key (wandb.ai/authorize): ')
wandb.login(key=wandb_key)

# ── 1. W&B ────────────────────────────────────────────────────
wandb.init(project='nalapro-swahili', name='task3-mlm-adaptation',
           config=CONFIG, reinit=True)

# Log MLM pre-training losses
for ep, loss in enumerate(mlm_history_afri):
    wandb.log({'mlm/afriberta_loss': loss, 'mlm/epoch': ep + 1})
for ep, loss in enumerate(mlm_history_swah):
    wandb.log({'mlm/swahbert_loss':  loss, 'mlm/epoch': ep + 1})

# Log classification metrics
for res in all_results:
    h = res['history']
    for i, ep in enumerate(h['epoch']):
        wandb.log({
            f"{res['tag']}/train_loss": h['train_loss'][i],
            f"{res['tag']}/val_acc":    h['val_acc'][i],
            f"{res['tag']}/val_f1":     h['val_f1'][i],
            'epoch': ep
        })
    wandb.log({
        f"{res['tag']}/test_acc":     res['test_acc'],
        f"{res['tag']}/test_f1":      res['test_f1'],
        f"{res['tag']}/best_val_acc": res['val_acc'],
        f"{res['tag']}/best_val_f1":  res['val_f1']
    })
    art = wandb.Artifact(f"model-{res['tag']}", type='model')
    art.add_file(res['model_path']); wandb.log_artifact(art)

wandb.log({
    'mlm_effect/afriberta_acc_gain_pct': afri_acc_gain,
    'mlm_effect/afriberta_f1_gain_pct':  afri_f1_gain,
    'mlm_effect/swahbert_acc_gain_pct':  swah_acc_gain,
    'mlm_effect/swahbert_f1_gain_pct':   swah_f1_gain
})
wandb.log({'task3_summary': wandb.Table(dataframe=results_df)})
wandb.log({'task3_chart':   wandb.Image('/content/task3_results.png')})
wandb_url = wandb.run.url
wandb.finish()

# ── 2. Aim ────────────────────────────────────────────────────
aim_run = aim.Run(experiment='nalapro-task3', repo='/content/.aim')
aim_run['hparams'] = CONFIG

for ep, loss in enumerate(mlm_history_afri):
    aim_run.track(loss, name='mlm_loss', epoch=ep+1, context={'model': 'AfriBERTa'})
for ep, loss in enumerate(mlm_history_swah):
    aim_run.track(loss, name='mlm_loss', epoch=ep+1, context={'model': 'SwahBERT'})

for res in all_results:
    h = res['history']
    for i, ep in enumerate(h['epoch']):
        aim_run.track(h['train_loss'][i], name='train_loss', epoch=ep, context={'method': res['tag']})
        aim_run.track(h['val_acc'][i],    name='val_acc',    epoch=ep, context={'method': res['tag']})
        aim_run.track(h['val_f1'][i],     name='val_f1',     epoch=ep, context={'method': res['tag']})
    aim_run.track(res['test_acc'], name='test_acc', context={'method': res['tag']})
    aim_run.track(res['test_f1'],  name='test_f1',  context={'method': res['tag']})

aim_run.track(afri_acc_gain, name='mlm_acc_gain_pct', context={'model': 'AfriBERTa'})
aim_run.track(afri_f1_gain,  name='mlm_f1_gain_pct',  context={'model': 'AfriBERTa'})
aim_run.track(swah_acc_gain, name='mlm_acc_gain_pct', context={'model': 'SwahBERT'})
aim_run.track(swah_f1_gain,  name='mlm_f1_gain_pct',  context={'model': 'SwahBERT'})
aim_run.close()

# ── 3. MLflow ────────────────────────────────────────────────
mlflow.set_experiment('nalapro-task3-mlm')
for res in all_results:
    with mlflow.start_run(run_name=f"task3-{res['tag']}"):
        mlflow.log_params({**CONFIG, 'method': res['tag']})
        h = res['history']
        for i, ep in enumerate(h['epoch']):
            mlflow.log_metric('train_loss', h['train_loss'][i], step=ep)
            mlflow.log_metric('val_acc',    h['val_acc'][i],    step=ep)
            mlflow.log_metric('val_f1',     h['val_f1'][i],     step=ep)
        mlflow.log_metric('test_acc',     res['test_acc'])
        mlflow.log_metric('test_f1',      res['test_f1'])
        mlflow.log_metric('best_val_acc', res['val_acc'])
        mlflow.log_metric('best_val_f1',  res['val_f1'])
        mlflow.log_artifact(res['model_path'])

with mlflow.start_run(run_name='task3-summary'):
    mlflow.log_metric('afriberta_acc_gain_pct', afri_acc_gain)
    mlflow.log_metric('afriberta_f1_gain_pct',  afri_f1_gain)
    mlflow.log_metric('swahbert_acc_gain_pct',  swah_acc_gain)
    mlflow.log_metric('swahbert_f1_gain_pct',   swah_f1_gain)
    mlflow.log_artifact('/content/task3_results.csv')
    mlflow.log_artifact('/content/task3_results.png')

print('\n' + '='*60)
print('✓ ALL TRACKING COMPLETE')
print('='*60)
print(f'W&B:    {wandb_url}')
print('Aim:    Run  !aim up --repo /content/.aim')
print('MLflow: Run  !mlflow ui --backend-store-uri /content/mlruns')
